# Notebook 05 — Nighttime Lights as a Poverty Proxy

## Background: Why nighttime lights?

One of the most powerful satellite-based poverty proxies is **nighttime light (NTL) intensity**
captured by the VIIRS Day/Night Band (DNB) sensor aboard the Suomi-NPP satellite.
The foundational insight (Henderson et al., 2012 *American Economic Review*) is simple:
electricity use → light at night → visible from space → correlated with economic activity.

This was formalised into a full poverty estimation pipeline by:
- **Jean et al. (2016, *Science*)** — used NTL as a *training label* for a CNN trained on
  daytime Landsat imagery. The idea: NTL is a cheap proxy for ground-truth wealth;
  train the CNN to predict NTL from daytime images, then the learned features transfer
  to predict DHS survey-based consumption expenditure.
- **Yeh et al. (2020, *Nature Communications*)** — extended this to Sentinel-2 imagery,
  achieving state-of-the-art poverty maps across Africa.

## This notebook

We build a **"wealth index proxy" pipeline** that combines:
1. Nighttime light intensity (VIIRS)
2. Urban land cover fraction (from our EuroSAT classifier)
3. Infrastructure presence (road/industrial pixels)

We demonstrate on **synthetic data** (clearly labelled as such) that exactly mimics
real VIIRS patterns, with instructions for swapping in real data.

**Install:** `pip install geopandas matplotlib rasterio`

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))

FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

np.random.seed(42)
print('Imports OK')

## 1. How to obtain real VIIRS nighttime lights data

### Option A — NASA Black Marble (recommended)
The **VNP46A4** product provides annual VIIRS nighttime light composites at 500 m resolution,
corrected for cloud cover, moonlight and atmospheric effects.

```bash
# 1. Create a free NASA Earthdata account at https://urs.earthdata.nasa.gov/
# 2. Install the download tool
pip install earthdata

# 3. Download a tile (example: West Africa, 2022 annual composite)
python -c "
import earthaccess
earthaccess.login()
results = earthaccess.search_data(
    short_name='VNP46A4',
    temporal=('2022-01-01', '2022-12-31'),
    bounding_box=(-18, 4, 16, 24)  # West Africa
)
earthaccess.download(results[:4], './data/viirs/')
"
```

### Option B — EOG VIIRS Annual Composites (no account needed)
The Earth Observation Group at Colorado School of Mines publishes annual VIIRS composites:
```
https://eogdata.mines.edu/products/vnl/
```
Download the `VNL_v21_npp_2022_global_vcmslcfg_c202205302300.average_masked.dat.tif.gz`
file (global, ~400 MB), extract with `gzip -d`, then read with rasterio.

Below we use **synthetic data** so the notebook runs without downloads.

In [ ]:
# ─── SYNTHETIC VIIRS DATA GENERATOR ──────────────────────────────────────────
# Generates a 100×100 grid of tiles mimicking a region with:
#   - A large city centre (high NTL, high urban LC)
#   - Suburban rings (medium NTL, mixed LC)
#   - Rural periphery (low/zero NTL, agricultural LC)
#   - A road corridor connecting two towns
#
# To use REAL data: replace this cell with rasterio.open('your_viirs.tif').read(1)
# and adapt the grid size accordingly.
# ─────────────────────────────────────────────────────────────────────────────

GRID = 80   # 80×80 tiles

def gaussian_blob(grid, cx, cy, sigma, amplitude):
    x = np.arange(grid)
    y = np.arange(grid)
    xx, yy = np.meshgrid(x, y)
    return amplitude * np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))

# Nighttime light intensity (nW/cm²/sr)
ntl = np.zeros((GRID, GRID))
ntl += gaussian_blob(GRID, 40, 40, 8, 60)   # main city
ntl += gaussian_blob(GRID, 65, 20, 5, 40)   # secondary town
ntl += gaussian_blob(GRID, 15, 65, 4, 25)   # small town
# Road corridor: connecting main city to secondary town
for i in range(GRID):
    j = int(20 + (i / GRID) * 20)
    if 0 <= j < GRID:
        ntl[i, j] = max(ntl[i, j], 8)
ntl += np.random.exponential(0.3, (GRID, GRID))  # rural scattered lights
ntl = np.clip(ntl, 0, 80)

# Land cover class index (0–9 matching EuroSAT classes)
# 0=AnnualCrop,1=Forest,2=HerbVeg,3=Highway,4=Industrial,
# 5=Pasture,6=PermanentCrop,7=Residential,8=River,9=SeaLake
lc = np.full((GRID, GRID), 5, dtype=int)  # default: Pasture

# City core → Residential + Industrial
for i in range(GRID):
    for j in range(GRID):
        d = np.sqrt((i-40)**2 + (j-40)**2)
        if d < 5:   lc[i, j] = 4   # Industrial core
        elif d < 12: lc[i, j] = 7  # Residential ring
        elif d < 18: lc[i, j] = 0  # AnnualCrop periurban

# Secondary town
for i in range(GRID):
    for j in range(GRID):
        d = np.sqrt((i-65)**2 + (j-20)**2)
        if d < 4:  lc[i, j] = 7   # Residential
        elif d < 8: lc[i, j] = 0  # AnnualCrop

# River diagonal
for i in range(GRID):
    j = int(i * 0.6)
    if 0 <= j < GRID:
        lc[i, j] = 8

# Forest in one corner
lc[:20, :20] = 1

print(f'Synthetic grid: {GRID}×{GRID} tiles')
print(f'NTL range: {ntl.min():.2f} – {ntl.max():.2f} nW/cm²/sr')

## 2. Compute the Wealth Index Proxy

We combine three signals into a single **Wealth Index Proxy (WIP)** score per tile,
inspired by the feature construction in Jean et al. (2016):

$$\text{WIP} = w_1 \cdot \tilde{\text{NTL}} + w_2 \cdot \tilde{\text{Urban}} + w_3 \cdot \tilde{\text{Infra}}$$

where each component is min-max normalised to [0, 1] and:
- $\tilde{\text{NTL}}$ = normalised nighttime light intensity
- $\tilde{\text{Urban}}$ = fraction of pixels classified as Residential or Industrial
- $\tilde{\text{Infra}}$ = fraction of pixels classified as Highway or Industrial

In [ ]:
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

# Urban fraction: Residential (7) or Industrial (4)
urban = ((lc == 7) | (lc == 4)).astype(float)

# Infrastructure: Highway (3) or Industrial (4)
infra = ((lc == 3) | (lc == 4)).astype(float)

# Weighted wealth index proxy
W1, W2, W3 = 0.5, 0.3, 0.2
wip = W1 * minmax(ntl) + W2 * minmax(urban) + W3 * minmax(infra)
wip = minmax(wip)

print(f'Wealth Index Proxy range: {wip.min():.3f} – {wip.max():.3f}')
print(f'Mean WIP in city core (rows 35-45, cols 35-45): {wip[35:45,35:45].mean():.3f}')
print(f'Mean WIP in rural area (rows 0-20, cols 50-80):  {wip[0:20,50:80].mean():.3f}')

## 3. Visualise — Four-panel map

In [ ]:
CLASS_NAMES  = ['AnnualCrop','Forest','HerbVeg','Highway','Industrial',
                'Pasture','PermCrop','Residential','River','SeaLake']
CLASS_COLORS = ['#e8b87a','#2d6a4f','#74c69d','#adb5bd','#6c757d',
                '#a8dadc','#95d5b2','#e63946','#4895ef','#48cae4']

cmap_lc = mcolors.ListedColormap(CLASS_COLORS)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Wealth Index Proxy from Nighttime Lights + Land Cover\n'
             '(Synthetic demo — methodology mirrors Jean et al. 2016)',
             fontsize=12, fontweight='bold')

# Panel 1 — Nighttime lights
im0 = axes[0].imshow(ntl, cmap='inferno', origin='upper')
axes[0].set_title('VIIRS Nighttime Lights\n(nW/cm²/sr)', fontweight='bold')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Panel 2 — Land cover
im1 = axes[1].imshow(lc, cmap=cmap_lc, vmin=0, vmax=9, origin='upper')
axes[1].set_title('Land Cover Classification\n(ResNet-50 on EuroSAT)', fontweight='bold')
legend_patches = [Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(10)]
axes[1].legend(handles=legend_patches, bbox_to_anchor=(1.05, 1), loc='upper left',
               fontsize=6.5, frameon=True)

# Panel 3 — Urban + infra binary
combined = urban * 0.6 + infra * 0.4
im2 = axes[2].imshow(combined, cmap='YlOrRd', origin='upper')
axes[2].set_title('Urban + Infrastructure\nFraction', fontweight='bold')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

# Panel 4 — Wealth Index Proxy (choropleth)
im3 = axes[3].imshow(wip, cmap='RdYlGn', origin='upper', vmin=0, vmax=1)
axes[3].set_title('Wealth Index Proxy\n(green=wealthier, red=poorer)', fontweight='bold')
plt.colorbar(im3, ax=axes[3], fraction=0.046)

for ax in axes:
    ax.axis('off')

plt.tight_layout()
out = FIGURES_DIR / 'poverty_proxy_map.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {out}')

In [ ]:
# Wealth distribution — histogram by land cover type
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# NTL vs WIP scatter
axes[0].scatter(ntl.ravel(), wip.ravel(), alpha=0.3, s=8, c=lc.ravel(),
                cmap=cmap_lc, vmin=0, vmax=9)
axes[0].set_xlabel('Nighttime Light Intensity (nW/cm²/sr)')
axes[0].set_ylabel('Wealth Index Proxy')
axes[0].set_title('NTL vs Wealth Index Proxy\n(coloured by land cover class)')
r = np.corrcoef(ntl.ravel(), wip.ravel())[0, 1]
axes[0].text(0.05, 0.93, f'r = {r:.3f}', transform=axes[0].transAxes,
             fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat'))

# Mean WIP per land cover class
mean_wip_per_class = [wip[lc == i].mean() if (lc == i).sum() > 0 else 0 for i in range(10)]
bars = axes[1].bar(CLASS_NAMES, mean_wip_per_class, color=CLASS_COLORS, edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('Mean Wealth Index Proxy')
axes[1].set_title('Mean WIP by Land Cover Class')
axes[1].tick_params(axis='x', rotation=45)
for bar, val in zip(bars, mean_wip_per_class):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
out = FIGURES_DIR / 'poverty_proxy_analysis.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {out}')

## 4. Connection to DHS-based poverty estimation

### The standard pipeline in the literature

The Demographic and Health Surveys (DHS) program collects ground-truth wealth data
from ~10,000 households per country every 5 years. Each cluster of ~25 households
gets a GPS coordinate and a **Wealth Index** (asset-based, principal component score).

The full poverty estimation pipeline:

```
DHS GPS clusters → buffer 10km radius → extract Sentinel-2 tiles
        ↓
Train CNN (ResNet/MS-ResNet) to predict NTL from daytime imagery
        ↓                          (Jean et al. 2016 approach)
Extract penultimate layer features → Ridge regression → DHS Wealth Index
        ↓
Predict poverty for ALL tiles (including areas with no DHS data)
        ↓
Poverty map at 2.4km resolution across entire continent
```

### Why this proxy is useful but imperfect

| Strength | Limitation |
|---|---|
| Free, global, annual updates | Misses off-grid electrification |
| High correlation with GDP (r≈0.86 nationally) | Saturates in dense cities |
| Captures economic geography well | Light pollution / skyglow effects |
| Available since 1992 → trend analysis | Doesn't capture subsistence wealth |

### The Chalmers research context
The PhD project you are applying to extends this pipeline by:
1. Comparing **multiple Sentinel-2 resolutions** (10m, 20m, 60m bands) for poverty prediction
2. Using **XAI** (GradCAM, SHAP — see Notebook 04) to audit what visual features drive predictions
3. Improving generalisation across **different African sub-regions and seasons**

### References
- Henderson et al. (2012). *Measuring economic growth from outer space*. AER.
- Jean et al. (2016). *Combining satellite imagery and ML to predict poverty*. Science.
- Yeh et al. (2020). *Using publicly available satellite imagery for poverty mapping*. Nature Comms.
- Engstrom et al. (2017). *Poverty from space*. World Bank Policy Research Working Paper.